# 캐시 마운트 검증

빌드한 캐시를 다른 커널에서 실제로 읽을 수 있는지 확인한다
CPU 전용이라 GPU 할당량을 쓰지 않는다

**핵심은 인덱스와 배열의 정렬이다**

`cache_index.csv` 가 "스터디 A = 샤드 0 의 3번 행" 이라고 적었는데
실제 3번 행이 다른 스터디면 이미지와 레이블이 어긋난 채로 학습하게 된다
에러는 안 나고 AUC 만 0.5 근처로 나온다


In [ ]:
import json, os, sys, time
from pathlib import Path

T0 = time.time()
for _c in ['/kaggle/input/datasets/hyejinkwak/knee-src', '/kaggle/input/knee-src']:
    if os.path.isdir(_c):
        sys.path.insert(0, _c); break
import paths, config, dicom_io
print(f'src import {time.time()-T0:.1f}s')
config.verify_labels()
print(paths.describe())


## 1. 마운트 경로 찾기

`kernel_sources` 로 붙인 커널 출력이 어디에 놓이는지 확인한다
재귀 glob 은 쓰지 않는다 (DICOM 819,640 개를 훑는다)


In [ ]:
t = time.time()
root = Path('/kaggle/input')
print('입력 최상위:', sorted(p.name for p in root.iterdir()))

CACHE = None
for d in [root/'knee-cache-build', root/'kernels'/'hyejinkwak'/'knee-cache-build']:
    if (d/'cache_meta.json').exists():
        CACHE = d; break
if CACHE is None:
    for d in list(root.glob('*/')) + list(root.glob('*/*/')) + list(root.glob('*/*/*/')):
        if (d/'cache_meta.json').exists():
            CACHE = d; break
assert CACHE is not None, '캐시를 못 찾음'
print(f'\n캐시 경로: {CACHE}   ({time.time()-t:.1f}s)')
print(sorted(p.name for p in CACHE.iterdir()))


In [ ]:
meta = json.loads((CACHE/'cache_meta.json').read_text())
print(json.dumps(meta, indent=2, ensure_ascii=False)[:600])

assert meta['slots'] == list(dicom_io.SLOTS), '슬롯이 현재 코드와 다르다'
assert len(meta['slots']) == 8 and meta['out_px'] == 224
print('\n설정 검증 통과')


## 2. 샤드 확인


In [ ]:
import numpy as np, pandas as pd

shards = sorted(CACHE.glob('cache_*.npy'))
tot = sum(f.stat().st_size for f in shards)/1024**3
print(f'샤드 {len(shards)}개  합계 {tot:.2f} GiB')
for f in shards:
    print(f'  {f.name}  {f.stat().st_size/1024**3:.2f} GiB')
assert len(shards) == meta['n_shards'], '샤드 수 불일치'

t = time.time()
arrs = [np.load(f, mmap_mode='r') for f in shards]
print(f'\nmmap open {time.time()-t:.2f}s')
for a, f in zip(arrs, shards):
    print(f'  {f.name}  {a.shape}  {a.dtype}')
assert sum(a.shape[0] for a in arrs) == meta['n_studies']
print('\n총 행 수 == 스터디 수 OK')


In [ ]:
# 메모리 사용량 - mmap 이 전체를 올리지 않는지
import resource
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2
print(f'최대 RSS {rss:.2f} GiB   (캐시 전체는 {tot:.2f} GiB)')
print('캐시보다 훨씬 작으면 mmap 이 제대로 동작하는 것이다')


## 3. 인덱스 <-> 배열 정렬 검증

**이 검증이 이 커널의 존재 이유다**

인덱스의 슬롯 채움 플래그와 배열에서 실제로 비어있는 칸을 대조한다
인덱스가 "3번 칸이 찼다" 는데 배열 3번 칸이 전부 0 이면 어긋난 것이다


In [ ]:
idx = pd.read_csv(CACHE/'cache_index.csv')
print('index', idx.shape)
print(idx.head(3).to_string())

assert len(idx) == meta['n_studies']
assert idx.study.is_unique, '스터디 UID 중복'

train = pd.read_csv(paths.TRAIN_CSV)
assert set(idx.study) == set(train.StudyInstanceUID), 'train.csv 와 스터디 집합이 다르다'
print('\n스터디 집합 일치 OK')


In [ ]:
SLOTS = list(dicom_io.SLOTS)
rng = np.random.default_rng(0)
sample = rng.choice(len(idx), 200, replace=False)

mismatch = []
for i in sample:
    r = idx.iloc[i]
    a = arrs[int(r.shard)][int(r.row)]          # (8, 9, 224, 224)
    nonzero = (a.reshape(len(SLOTS), -1).max(axis=1) > 0)
    claimed = np.array([bool(r[s]) for s in SLOTS])
    if not np.array_equal(nonzero, claimed):
        bad = [SLOTS[k] for k in range(len(SLOTS)) if nonzero[k] != claimed[k]]
        mismatch.append((r.study[-12:], bad))

print(f'표본 {len(sample)}건 중 불일치 {len(mismatch)}건')
for m in mismatch[:10]:
    print('  !', m)
assert not mismatch, '인덱스와 배열이 어긋난다'
print('\n인덱스 <-> 배열 정렬 검증 통과')


In [ ]:
# 전 샤드 경계 확인 - 샤드마다 첫 행과 마지막 행을 본다
print('샤드 경계 검사')
bad = 0
for si, a in enumerate(arrs):
    for row in (0, a.shape[0]-1):
        r = idx[(idx.shard == si) & (idx.row == row)]
        assert len(r) == 1, f'shard {si} row {row} 인덱스 없음'
        r = r.iloc[0]
        nz = (np.asarray(a[row]).reshape(len(SLOTS), -1).max(axis=1) > 0)
        cl = np.array([bool(r[s]) for s in SLOTS])
        ok = np.array_equal(nz, cl)
        bad += (not ok)
        print(f'  shard {si} row {row:>3}  {"OK" if ok else "MISMATCH"}  {r.study[-10:]}')
assert bad == 0
print('\n샤드 경계 OK')


## 4. 이미지 육안 확인


In [ ]:
import matplotlib.pyplot as plt

OUT = Path('/kaggle/working')
pick = rng.choice(len(idx), 2, replace=False)
for p in pick:
    r = idx.iloc[p]
    a = np.asarray(arrs[int(r.shard)][int(r.row)])
    fig, axes = plt.subplots(len(SLOTS), 9, figsize=(11, len(SLOTS)*1.35))
    for i, slot in enumerate(SLOTS):
        for j in range(9):
            ax = axes[i, j]
            ax.imshow(a[i, j], cmap='gray', vmin=0, vmax=255)
            ax.set_xticks([]); ax.set_yticks([])
            if j == 0:
                ax.set_ylabel(slot, fontsize=7,
                              color='black' if r[slot] else 'red')
    fig.suptitle(f'shard {r.shard} row {r.row}  {r.study[-16:]}  '
                 f'{r.n_filled}/8 slots', fontsize=10)
    plt.tight_layout()
    fig.savefig(OUT/f'verify_{r.shard}_{r.row}.png', dpi=90, bbox_inches='tight')
    plt.show()


## 5. 학습 커널이 쓸 읽기 속도


In [ ]:
t = time.time()
N = 64
sel = rng.choice(len(idx), N, replace=False)
batch = np.stack([np.asarray(arrs[int(idx.iloc[i].shard)][int(idx.iloc[i].row)])
                  for i in sel])
el = time.time() - t
print(f'{N}건 랜덤 읽기 {el:.2f}s  ({el/N*1000:.1f} ms/study)')
print(f'배치 {batch.shape}  {batch.nbytes/1024**2:.0f} MiB')
print(f'1 epoch (4,407건) 읽기만 {el/N*len(idx)/60:.1f}분')

summary = {'cache_path': str(CACHE), 'shards': len(shards),
           'total_gib': round(tot, 2), 'mmap_open_s': None,
           'index_rows': int(len(idx)), 'mismatch': len(mismatch),
           'ms_per_study': round(el/N*1000, 1),
           'max_rss_gib': round(rss, 2),
           'total_s': round(time.time()-T0, 1)}
(OUT/'verify_report.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))
